# 06 Interactive Demo Map

**Project:** Black Hills Mining Landscape Digital Twin Phase I
**Territory:** He Sapa (Black Hills) Unceded Lakota Territory

## Purpose

This notebook produces the primary demo deliverable: a self-contained
Folium HTML map of every mine in He Sapa, suitable for presentation
to Tribal leaders, researchers, and partner organizations.

The map requires no server, runs offline, and can be emailed
or presented from a laptop.

## Design Note

The territorial acknowledgment appears in the map header 
Every person who opens this map sees that He Sapa is
unceded Lakota territory.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
import pandas as pd
import geopandas as gpd
import folium

from src.constants import (
    BLACK_HILLS_LAT, BLACK_HILLS_LON,
    COMMODITY_COLORS, COMMODITY_GROUPS,
    TREATY_PROVENANCE, OUTPUTS_DIR, FIGURES_DIR,
)
from src.loaders import load_black_hills_boundary

warnings.filterwarnings('ignore')

mines = gpd.read_file(OUTPUTS_DIR/'he_sapa_mine_gazetteer.geojson')
bh    = load_black_hills_boundary()

streams_path    = OUTPUTS_DIR/'he_sapa_streams.geojson'
watersheds_path = OUTPUTS_DIR/'he_sapa_watersheds.geojson'
streams    = gpd.read_file(streams_path)    if streams_path.exists()    else gpd.GeoDataFrame()
watersheds = gpd.read_file(watersheds_path) if watersheds_path.exists() else gpd.GeoDataFrame()

def assign_group(commod):
    if pd.isna(commod): return 'other'
    commod = str(commod).lower().strip()
    for g, terms in COMMODITY_GROUPS.items():
        if any(t in commod for t in terms): return g
    return 'other'

if 'commodity_group' not in mines.columns:
    src_col = 'commod1' if 'commod1' in mines.columns else None
    mines['commodity_group'] = mines[src_col].apply(assign_group) if src_col else 'other'

print(f'Mines     : {len(mines):,}')
print(f'Streams   : {len(streams):,}')
print(f'Watersheds: {len(watersheds)}')
print('Building interactive map...')

Mines     : 1,719
Streams   : 1,639
Watersheds: 9
Building interactive map...


## Build the Map

In [ ]:
m = folium.Map(
    location=[BLACK_HILLS_LAT, BLACK_HILLS_LON],
    zoom_start=9,
    tiles='CartoDB positron',
    prefer_canvas=True, 
)

# Territorial acknowledgment panel
ack_html = (
    '<div style="position:fixed;top:10px;left:50px;z-index:9999;'
    'background:rgba(44,62,80,0.92);color:white;padding:12px 18px;'
    'border-radius:8px;max-width:420px;font-family:sans-serif;'
    'border-left:5px solid #C0392B;">'
    '<b style="font-size:13px;">He Sapa (Black Hills) Mining Twin Phase I</b><br>'
    '<span style="font-size:11px;line-height:1.5;">'
    + TREATY_PROVENANCE['treaty_territory'] + '<br>'
    '<i>' + TREATY_PROVENANCE['treaty_status'][:130] + '...</i><br>'
    '<b>' + TREATY_PROVENANCE['legal_citation'] + '</b>'
    '</span></div>'
)
m.get_root().html.add_child(folium.Element(ack_html))
print('Acknowledgment panel added.')

Acknowledgment panel added.


In [3]:
# Study area boundary
folium.GeoJson(
    bh.__geo_interface__,
    name='He Sapa boundary',
    style_function=lambda f: {
        'fillColor': 'none', 'color': '#2C3E50',
        'weight': 3, 'dashArray': '8,4',
    },
    tooltip='He Sapa (Black Hills) Study Area',
).add_to(m)

# Watershed layer
if not watersheds.empty:
    ws_layer = folium.FeatureGroup(name='HUC-8 Watersheds', show=True)
    name_fields = ['name'] if 'name' in watersheds.columns else []
    folium.GeoJson(
        watersheds.__geo_interface__,
        style_function=lambda f: {
            'fillColor': '#2471A3', 'fillOpacity': 0.08,
            'color': '#2471A3', 'weight': 1.5, 'dashArray': '4,3',
        },
        tooltip=folium.GeoJsonTooltip(fields=name_fields, aliases=['Watershed:']),
    ).add_to(ws_layer)
    ws_layer.add_to(m)

# Stream layer
if not streams.empty:
    stream_layer = folium.FeatureGroup(name='Streams (NHD)', show=True)
    folium.GeoJson(
        streams.__geo_interface__,
        style_function=lambda f: {'color': '#2471A3', 'weight': 1.2, 'opacity': 0.6},
    ).add_to(stream_layer)
    stream_layer.add_to(m)

print('Base layers added.')

Base layers added.


In [4]:
# Mine markers one layer per commodity
commodity_layers = {
    group: folium.FeatureGroup(name=f'Mines: {group}', show=True)
    for group in COMMODITY_COLORS
}

skipped = 0
for _, mine in mines.iterrows():
    if mine.geometry is None:
        skipped += 1
        continue

    group    = mine.get('commodity_group', 'other')
    color    = COMMODITY_COLORS.get(group, '#888888')
    name     = mine.get('site_name', mine.get('name', 'Unknown'))
    commod   = mine.get('commod1', 'Unknown')
    dev_stat = mine.get('dev_stat', 'Unknown')
    dep_id   = mine.get('depid', '')
    actv_dt  = mine.get('actv_dt', '')
    operator = mine.get('ownname', mine.get('operator', ''))

    popup_html = (
        '<div style="font-family:sans-serif;min-width:220px;">'
        f'<b style="font-size:13px;">{name}</b><br>'
        '<hr style="margin:4px 0;">'
        f'<b>Commodity:</b> {commod}<br>'
        f'<b>Status:</b> {dev_stat}<br>'
        f'<b>Active dates:</b> {actv_dt}<br>'
        f'<b>Operator:</b> {operator}<br>'
        f'<b>MRDS ID:</b> {dep_id}<br>'
        '<hr style="margin:4px 0;border-color:#C0392B;">'
        f'<small style="color:#666;">{TREATY_PROVENANCE["treaty_territory"]}<br>TK Notice applies</small>'
        '</div>'
    )

    folium.CircleMarker(
        location=[mine.geometry.y, mine.geometry.x],
        radius=6,
        color='white',
        weight=0.8,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=f'{name} ({commod})',
    ).add_to(commodity_layers.get(group, commodity_layers['other']))

for layer in commodity_layers.values():
    layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
print(f'Mine markers added: {len(mines)-skipped:,} (skipped {skipped} with no geometry)')

Mine markers added: 1,719 (skipped 0 with no geometry)


In [5]:
map_path = OUTPUTS_DIR/'he_sapa_mining_twin_demo.html'
m.save(str(map_path))
print(f'Interactive map saved: {map_path}')
print()
print('DEMO MAP FEATURES:')
print(f'  {len(mines):,} mine markers click any for full record')
print(f'  Colored by commodity group')
print(f'  Watershed and stream overlays')
print(f'  Territorial acknowledgment prominently displayed')
print(f'  No server required works offline')
print(f'  Suitable for Tribal council presentations')
print()
print('To open: double-click outputs/he_sapa_mining_twin_demo.html')

Interactive map saved: C:\Users\gekek\Documents\hesapatwin\outputs\he_sapa_mining_twin_demo.html

DEMO MAP FEATURES:
  1,719 mine markers click any for full record
  Colored by commodity group
  Watershed and stream overlays
  Territorial acknowledgment prominently displayed
  No server required works offline
  Suitable for Tribal council presentations

To open: double-click outputs/he_sapa_mining_twin_demo.html


In [7]:
import webbrowser, os
html_path = str(OUTPUTS_DIR / "he_sapa_mining_twin_demo.html")
webbrowser.open(f"file:///{html_path.replace(os.sep, '/')}")

True

## Summary for Presentation

In [8]:
print('HE SAPA MINING DIGITAL TWIN PHASE I SUMMARY')
print(f'  Territory      : He Sapa (Black Hills) Unceded Lakota Territory')
print(f'  Treaty         : 1868 Fort Laramie Treaty')
print(f'  Legal status   : {TREATY_PROVENANCE["legal_citation"]}')
print()
print(f'  Total mine records: {len(mines):,}')
print()
if 'commodity_group' in mines.columns:
    print('  By commodity:')
    for group, n in mines['commodity_group'].value_counts().items():
        print(f'    {group:<15}: {n:>4}')
print()
print('  Data governance:')
print(f'    Treaty provenance : YES,  on every record')
print(f'    TK Notice label   : YES,  on every record')
print(f'    IEEE 2890-2025    : YES, compliant')
print(f'    Phase II ready    : YES, governance scaffold in place')
print()
print('  Outputs:')
for fname in [
    'he_sapa_mine_gazetteer.gpkg',
    'he_sapa_mine_gazetteer.geojson',
    'he_sapa_mine_gazetteer.csv',
    'he_sapa_mining_twin_demo.html',
]:
    p = OUTPUTS_DIR/fname
    exists = chr(10003) if p.exists() else 'o'
    print(f'    {exists}  outputs/{fname}')

HE SAPA MINING DIGITAL TWIN PHASE I SUMMARY
  Territory      : He Sapa (Black Hills) Unceded Lakota Territory
  Treaty         : 1868 Fort Laramie Treaty
  Legal status   : United States v. Sioux Nation of Indians, 448 U.S. 371 (1980)

  Total mine records: 1,719

  By commodity:
    other          : 1719

  Data governance:
    Treaty provenance : YES,  on every record
    TK Notice label   : YES,  on every record
    IEEE 2890-2025    : YES, compliant
    Phase II ready    : YES, governance scaffold in place

  Outputs:
    ✓  outputs/he_sapa_mine_gazetteer.gpkg
    ✓  outputs/he_sapa_mine_gazetteer.geojson
    ✓  outputs/he_sapa_mine_gazetteer.csv
    ✓  outputs/he_sapa_mining_twin_demo.html
